In [31]:
# use float64'
from jax import config
config.update("jax_enable_x64", True)

In [32]:
import jax
import jax.numpy as jnp
import jax.random as jr

import matplotlib.pyplot as plt

import os
import requests
# import sys
# sys.path.append('../')
import pickle

import pickle
import pandas as pd
import json
import numpy as np
from pathlib import Path

from hybrid_models.utils import Dataset
from scipy.interpolate import CubicSpline
from scipy.integrate import quad

DATA_DIR = Path("../../data/icg_channels/icg_pickles")

In [33]:
# Omni model definitions and "fitting config"
# taken from https://github.com/icgenealogy/icg-channels/blob/master/icg-scripts/fitting_supermodel.py

def sigmoid(x, a, b):
    return 1 / (1 + jnp.exp(-a * x + b))

def modified_sigmoid(x, a, b, c, d):
    return c / (1 + jnp.exp(-a * x + b)) + d

def tau_fun1(x, a, b, c, d):
    y = x - a
    return b / (jnp.exp(-(c * y)) + jnp.exp(d * y))

def tau_fun2(x, a, b, c, d, e, f):
    y = x - a
    return b / (jnp.exp(-(c * y + d * y**2)) + jnp.exp(e * y + f * y**2))

def tau_fun3(x, a, b, c, d, e, f, g, h):
    y = x - a
    return b / (
        jnp.exp(-(c * y + d * y**2 + e * y**3))
        + jnp.exp(f * y + g * y**2 + h * y**3)
    )

def tau_fun4(x, a, b, c, d, e, f, g, h, i, j):
    y = x - a
    return b / (
        jnp.exp(-(c * y + d * y**2 + e * y**3 + f * y**4))
        + jnp.exp(g * y + h * y**2 + i * y**3 + j * y**4)
    )

def err1(data, fun, v, params):
    data_arr = jnp.asarray(data)
    pred_arr = jnp.asarray(fun(v, *params))
    rmse = jnp.sqrt(jnp.mean((data_arr - pred_arr) ** 2))
    stdev = jnp.std(data_arr)
    return rmse / stdev

def err2(data, fun, v, params):
    P = CubicSpline(jnp.asarray(v), jnp.asarray(data))

    def area_diff_fun(x):
        return (P(x) - fun(x, *params)) ** 2

    def area_fun(x):
        return P(x) ** 2

    x = quad(area_diff_fun, v[0], v[-1])
    x_norm = quad(area_fun, v[0], v[-1])
    return x[0] / x_norm[0]

def err3(data, fun, v, params):
    data_arr = jnp.asarray(data)
    pred_arr = jnp.asarray(fun(v, *params))
    return jnp.linalg.norm(data_arr - pred_arr) / jnp.linalg.norm(data_arr)

SM_NUM = 2
# ## IMPORTANT: when specifying more than one fit function for steady-state or tau
# ## ss_transitions and tau_transitions: specifies how parameters of previous fit function relate to current one
# ## ss_identity and tau_identity: specifies how parameters of each fit function relate to the final one

# if SM_NUM==1:
#     ss_fit_fcn_list = [sigmoid]  # NOTE: supermodel fcns should be in the form of a list
#     n_ss_params = [2]
#     ss_identity = [np.arange(n_ss_params[-1])]
#     tau_fit_fcn_list = [tau_fun1, tau_fun2, tau_fun3]
#     n_tau_params = [4, 6, 8]
#     tau_transitions = [[0, 1, 2, 4], [0, 1, 2, 3, 5, 6]]
#     tau_identity = [[0, 1, 2, 5], [0, 1, 2, 3, 5, 6],
#                     np.arange(n_tau_params[-1])]

# elif SM_NUM==2:
#     ss_fit_fcn_list = [sigmoid, modified_sigmoid]
#     n_ss_params = [2, 4]
#     ss_transitions = [[0, 1]]
#     ss_identity = [[0, 1], np.arange(n_ss_params[-1])]
#     tau_fit_fcn_list = [tau_fun1, tau_fun2, tau_fun3]
#     n_tau_params = [4, 6, 8]
#     tau_transitions = [[0, 1, 2, 4], [0, 1, 2, 3, 5, 6]]
#     tau_identity = [[0, 1, 2, 6], [0, 1, 2, 3, 5, 6],
#                     np.arange(n_tau_params[-1])]

# elif SM_NUM==3:
#     ss_fit_fcn_list = [sigmoid]
#     n_ss_params = [2]
#     ss_identity = [np.arange(n_ss_params[-1])]
#     tau_fit_fcn_list = [tau_fun1, tau_fun2]
#     n_tau_params = [4, 6]
#     tau_transitions = [[0, 1, 2, 4]]
#     tau_identity = [[0, 1, 2, 4], np.arange(n_tau_params[-1])]

# elif SM_NUM==4:
#     ss_fit_fcn_list = [sigmoid]
#     n_ss_params = [2]
#     ss_identity = [np.arange(n_ss_params[-1])]
#     tau_fit_fcn_list = [tau_fun1]
#     n_tau_params = [4]
#     tau_identity = [np.arange(n_tau_params[-1])]

# elif SM_NUM==5:
#     ss_fit_fcn_list = [sigmoid]
#     n_ss_params = [2]
#     ss_identity = [np.arange(n_ss_params[-1])]
#     tau_fit_fcn_list = [tau_fun1, tau_fun2, tau_fun3, tau_fun4]
#     n_tau_params = [4, 6, 8, 10]
#     tau_transitions = [[0, 1, 2, 4], [0, 1, 2, 3, 5, 6],
#                        [0, 1, 2, 3, 4, 6, 7, 8]]
#     tau_identity = [[0, 1, 2, 6], [0, 1, 2, 3, 6, 7], [0, 1, 2, 3, 4, 6, 7, 8],
#                     np.arange(n_tau_params[-1])]

omni_models = {
    "SM1_SS": sigmoid, "SM1_TAU": tau_fun3,
    "SM2_SS": modified_sigmoid, "SM2_TAU": tau_fun3,
    "SM3_SS": sigmoid, "SM3_TAU": tau_fun2,
    "SM4_SS": sigmoid, "SM4_TAU": tau_fun1,
    "SM5_SS": sigmoid, "SM5_TAU": tau_fun4,
    }

In [5]:
PICKLES_URL = "https://api.github.com/repos/icgenealogy/icg-channels/contents/icg-pickles"

response = requests.get(PICKLES_URL)
response.raise_for_status()
files = response.json()

for file in files:
    if file["name"].endswith(".pkl"):
        download_url = file["download_url"]
        filename = file["name"]
        filepath = os.path.join(DATA_DIR, filename)

        print(f"Downloading {filename}...")

        with requests.get(download_url, stream=True) as r:
            r.raise_for_status()
            with open(filepath, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

In [39]:
TEMPS = [37, 23, 6.3]
dummy_as = lambda x: jnp.full_like(x, np.nan)
is_list = lambda x: isinstance(x, list)
as_array = lambda x: jnp.array(x)

v_grid = jnp.array([-100.,  -90., *list(range(-80, 10, 2)), *list(range(10, 80, 10)), 80., 100.])
nans = jnp.full_like(v_grid, jnp.nan)
omni_names = [f"SM{i}" for i in range(1, 6)]

num_channels_total = 0
files = os.listdir(DATA_DIR)
for num, fname in enumerate(files):
    with open(f"{DATA_DIR}/{fname}", "rb") as f:
        icg_channels = pickle.load(f)

    fname_no_ext = os.path.splitext(fname)[0]
    dir_path = os.path.join(DATA_DIR.parent, fname_no_ext)
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)

    counter = 0
    for name in icg_channels:
        X, Y = {"v": {}}, {"tau": {}, "xinf": {}}
        METADATA = {"omni": {}}
        channel = icg_channels[name]
        
        if len(channel["ERROR_FLAGS"]) == 0:
            valid_states = list(channel['GATES'].keys())
            valid_states = [k for k in valid_states if len(channel["RATE_VALS_TAU"][k]) > 0 and len(channel["RATE_VALS_SS"][k]) > 0]
            if len(valid_states) > 0:
                X["v"] = {k: v_grid for k in valid_states}
                Y["tau"] = {k: jnp.array(channel["RATE_VALS_TAU"][k]) for k in valid_states}
                Y["xinf"] = {k: jnp.array(channel["RATE_VALS_SS"][k]) for k in valid_states}

                METADATA["fit"] = False
                for omni_key in omni_names:
                    METADATA["omni"][omni_key] = {"xinf": {}, "tau": {}}
                    if any([channel[f"{omni_key}_FIT"] for omni_key in omni_names]):
                        METADATA["fit"] = True
                    
                    vx = X["v"]
                    params_ss = channel[f"{omni_key}_PARAMS_SS"]
                    params_tau = channel[f"{omni_key}_PARAMS_TAU"]
                    for state in valid_states:
                        omni_ss = omni_models[omni_key + "_SS"]
                        omni_tau = omni_models[omni_key + "_TAU"]

                        ss_omni_vals = dummy_as(vx[state])
                        if len(params_ss[state]) > 0:
                            ss_omni_vals = omni_ss(vx[state], *params_ss[state])
                        
                        tau_omni_vals = dummy_as(vx[state])
                        if len(params_tau[state]) > 0:
                            tau_omni_vals = omni_tau(vx[state], *params_tau[state])
                        
                        METADATA["omni"][omni_key]["xinf"][state] = ss_omni_vals
                        METADATA["omni"][omni_key]["tau"][state] = tau_omni_vals
                            
                METADATA.update({f"powx['{k}']": channel["GATES"][k] for k in valid_states})

                METADATA["states"] = valid_states
                METADATA["gx"] = channel["G_VALS"]

                data = Dataset(X, Y, METADATA)
                data.to_file(f"{dir_path}/{name.replace(".mod", "")}.json")
                counter += 1
                num_channels_total += 1
    print(f"Saved {counter} channels out of {len(icg_channels)} from {fname} {num+1}/{len(files)}")
print(f"Total channels processed: {num_channels_total}")

Saved 94 channels out of 263 from icg-channels-KCa.pkl 1/5
Saved 442 channels out of 633 from icg-channels-Ca.pkl 2/5
Saved 167 channels out of 250 from icg-channels-IH.pkl 3/5
Saved 1104 channels out of 1455 from icg-channels-K.pkl 4/5
Saved 655 channels out of 923 from icg-channels-Na.pkl 5/5
Total channels processed: 2462
